# For developers

smICA analytical methods are directory-based add-ons. Each complete method placed in `src/smICA/Modes` is discovered at startup and added to the **Analysis** menu. The built-in `Extract_from_PTU` and `Phot2conc` methods use the same mechanism as additional methods.

## Start from the template

1. Copy the complete `templates/add_on_template` directory to `src/smICA/Modes`.
2. Rename the copied directory to a short, import-safe method name.
3. Rename the files and replace every `Template` and `template` identifier, including import paths, class names, callback names, mode paths, Dear PyGui tags, and visible labels.
4. Keep the method self-contained inside its directory. Shared, generally useful code may be placed under `src/smICA/Required` only when it is intentionally part of the application API.
5. Start smICA from a source checkout and confirm that the new entry appears in the **Analysis** menu, mounts correctly, resizes, and can be replaced by each other mode without leaving widgets behind.

The template is executable after it is copied without renaming, but all identifiers must be made unique before another add-on is introduced.

## Required directory contract

A mode directory is accepted only when it contains `__init__.py`, exactly one `*_config.json` file, and all three Python files named by that configuration. The configuration has four required string fields:

```json
{
  "LAYOUT": "MyMethod_layout.py",
  "INIT": "MyMethod_INIT.py",
  "ANAL_MENU_ITEM": "MyMethod_menu_item.py",
  "menu_class_func": "MyMethod_menu_F"
}
```

The filenames must be local basenames rather than absolute paths or paths containing parent-directory components. An incomplete or malformed method is ignored, which prevents a partially copied add-on from breaking application startup.

## Responsibilities of the files

- The `*_menu_item.py` file creates one item under `menu_analysis_method_dropout`. Its controller unmounts the active mode, sets `inV.mounted_method`, executes the selected layout, and connects mode-specific File-menu callbacks.
- The `*_layout.py` file creates Dear PyGui windows and widgets. Every created item that must disappear while switching methods must be appended to `globalITEMS.windows`. It also installs the method's resize callback.
- The `*_INIT.py` file owns layout geometry, state, callbacks, validation, and analytical functions. Keeping calculations outside the layout declarations makes them easier to test.
- The config file connects these entry points without requiring edits to the central application launcher.

Dear PyGui tags are application-global. Prefix every tag with the method name. Delete owned items in reverse creation order, reset the viewport resize callback when unmounting, and never add the Analysis-menu selector itself to `globalITEMS.windows`.

## Data and calculation design

Separate file loading, numerical calculations, GUI updates, and exports into small functions. Numerical functions should accept arrays or plain Python values and return results without reading widget state internally; callbacks can translate GUI values into those inputs. Validate dimensionality, units, empty selections, NaN values, and incompatible channel metadata before starting a long calculation. Use `os.path` for filesystem paths so the method works on Linux and Windows.

Add required third-party libraries to the main project dependency definition. An updater can install code distributed by the official repository, but an add-on must not download or execute an undeclared dependency by itself.

## Updater behavior

During an official update, smICA scans `src/smICA/Modes` in the downloaded repository. Every complete mode is copied into the installed application's `Modes` directory. Existing official modes are updated and new official add-ons are installed. Local mode directories that are absent from the repository are preserved; the synchronizer does not delete them. Incomplete repository mode directories are skipped and reported in the local update output. Restart smICA after an update so discovery runs again.

Add-ons execute Python code inside the smICA process and have the same filesystem permissions as the application. Install only trusted add-ons and review their source before copying them into `Modes`.

## Testing and automation

Add repository-level numerical and lifecycle tests under `dev_tests`; do not place developer-only tests in the installed calculation-test suite. At minimum, test calculation functions independently, missing or invalid input handling, mounting and unmounting, mode switching, resizing, and Linux/Windows path handling. Run the developer suite from the repository root with:

```bash
python dev_tests/run_tests.py
```

If a method must be controlled through the TCP server, register narrowly scoped public commands in the automation command layer. A command should call the same callback path as the matching GUI action, validate its preconditions before changing state, return JSON-compatible results, and expose no arbitrary Python execution or unrestricted filesystem operation. Document each public command in the automation command reference.

## Pre-merge checklist

- The directory passes the required config contract and is discovered in deterministic order.
- All files use the project MIT header and all source comments and identifiers are in English.
- GUI tags, classes, callbacks, and menu objects are unique.
- Switching to and from every installed mode removes all owned GUI items.
- Long operations provide progress feedback and leave the GUI responsive.
- Paths are built with `os.path` and have been checked on Linux and Windows.
- Dependencies, user-facing behavior, outputs, and automation commands are documented.
- `python dev_tests/run_tests.py` passes.